# Feature Engineering

## Objective

The goal of this notebook is to create new features from the cleaned dataset that can improve the recommendation system.

Instead of relying on the metadata provided by Open Opus API, additional musicl features will be extracted from the work titles and subtitles. These engineered features will provide a richer representation of each musical work and improve similarity-based recommedations.

The following featured will be created:
- Title + subtitle
- Musical form
- Extended Genre
- Instrumentation (when available)
- Musical key (when available)
- Tempo (when available)

The processed dataset generated in this notebook will be used for exploratory data analysis and recommendation model development.

In [1]:
import pandas as pd
from pathlib import Path
from src.feature_extraction import (extract_form, extract_extended_genre,
                                    extract_instruments, extract_mode, extract_tempo)

In [2]:
df = pd.read_csv(Path('../data/processed/works_with_composers_cleaned.csv'))

In [3]:
df.columns

Index(['title', 'subtitle', 'popular', 'recommended', 'work_id', 'genre',
       'composer_id', 'composer_name', 'birth', 'death', 'epoch'],
      dtype='str')

### Create Title and Subtitle Feature

The new feature **title_subtitle** is created to combine both title feature and subtitle feature to extract musical forms, genres and instruments.

In [4]:
df['title_subtitle'] = (df['title'].fillna("") + " ") + (df['subtitle'].fillna("").str.strip())

### Extract Musical Form Feature

A rule-based approach was used to extract musical forms from combined **title** and **subtitle** fields. Common forms, spelling variations and multilingual names were standardized into a consistent set of categories.

In [5]:
df['form'] = df['title_subtitle'].apply(extract_form)

In [6]:
df['form'].value_counts()

form
Sonata         1200
Concerto        746
Symphony        500
Quartet         383
Trio            247
Suite           241
Fugue           188
Variations      160
Prelude         114
Overture        102
Waltz            90
Fantasy          80
Minuet           77
Duet             70
Romance          68
Quintet          63
Serenade         51
Rondo            49
Scherzo          46
Canon            44
Capriccio        37
Polonaise        33
Mazurka          31
Ballade          29
Ordre            28
Nocturne         27
Impromptu        25
Lullaby          24
Polka            24
Toccata          23
Partita          19
Rhapsody         16
Etude            14
Chaconne         13
Intermezzo       11
Passacaglia      10
Sarabande         9
Invention         4
Interlude         3
Arabesque         2
Name: count, dtype: int64

#### Form Extraction Report

Musical forms were successfully identified for approximately 5000 of the 13000 works. The remaining works either do not explicitly mention their form in the available metadata or use titles that do not provide sufficient information for rule-based extraction.

### Extract Extended Genre Feature

The original dataset provides only a small set of broad genre categories. To create a more informative feature, an **extended_genre** column was generated by extracting more specific work types from the **title** and **subtitle** fields.

Examples include categories such as Opera, Mass, Cantata, Oratorio, Passion and Ballet. When no additional genre information could be extracted the original genre value from the dataset was retained.

In [7]:
df['extended_genre'] = df[['title_subtitle', 'genre']].apply(extract_extended_genre, axis=1)

In [8]:
df['extended_genre'].value_counts()

extended_genre
Vocal               3850
Keyboard            2862
Orchestral          1998
Chamber             1843
Song                 776
Opera                354
Aria                 328
Cantata              266
Stage                229
Lied                 130
Ballet                98
Incidental Music      55
Mass                  46
Lieder                36
Oratorio              29
Musical               29
Salve Regina          28
Choral                26
Passion               16
Stabat Mater          14
Operetta               1
Name: count, dtype: int64

### Extracting Musical Instruments

A rule-based approach will be used to extract the primary musical instruments or performing forces from the available metadata. Instrument information is first identified from the **title** and **subtitle** fields. When no explicit instrument is found the **extended genre** feature is used to infer the most likely performing forces.

In [9]:
df['instrumentation'] = df[['title_subtitle', 'extended_genre', 'form']].apply(extract_instruments, axis=1)

In [10]:
df['instrumentation'].value_counts()

instrumentation
Voice                                          3382
Piano                                          3162
Orchestra                                      1572
ChamberEnsemble                                 814
Voice, Piano                                    743
                                               ... 
Violin, Oboe, Strings, Cello, Orchestra           1
Violin, Oboe, Strings, Orchestra, Orchestra       1
Violin, Oboe, Clarinet, Orchestra                 1
Violin, Clarinet, Chorus, Guitar                  1
Clarinet, Guitar                                  1
Name: count, Length: 197, dtype: int64

### Extract Musical Modality

A new feature will be created to capture the character of each work. Mode in music such as major or minor determines the overall character and emotion of a piece. On the other hand the tonality of a work doesn't give us any meaningful information so only the modality of a musical piece is extracted.

In [11]:
df['mode'] = df['title_subtitle'].apply(extract_mode)

In [12]:
df['mode'].value_counts()

mode
Major       2393
Minor       1202
Phrygian       2
Name: count, dtype: int64

### Extract Tempo Feature

Tempo information is extracted from the **title** and **subtitle** fields by identifying common classical tempo markings. This feature can give us information about the character and mood of a work.

In [13]:
df['tempo_marking'] = df['title_subtitle'].apply(extract_tempo)

In [14]:
df['tempo_marking'].value_counts()

tempo_marking
Andante       52
Allegro       46
Adagio        39
Allegretto    32
Moderato       9
Presto         9
Lento          8
Largo          6
Vivace         3
Name: count, dtype: int64

### Overview of Dataset with New Features

In [15]:
df[['title', 'composer_name', 'form', 'extended_genre', 'instrumentation', 'tempo_marking']].sample(10)

,title,composer_name,form,extended_genre,instrumentation,tempo_marking
10431,"Der Vater mit dem Kind, D.906",Franz Schubert,NaN,Vocal,Voice,NaN
11819,Canticum Sacrum,Igor Stravinsky,NaN,Cantata,"Voice, Chorus, Orchestra",NaN
10904,"Trinklied, D.75, ""Freunde, sammelt euch im Kri...",Franz Schubert,NaN,Lied,"Voice, Piano",NaN
9159,"6 Etudes, op. 111",Camille Saint-Saëns,NaN,Keyboard,Piano,NaN
3586,"6 Songs, op. 25",Edvard Grieg,NaN,Song,"Voice, Piano",NaN
12344,"Bassoon Concerto in B flat major, RV.501, ""La ...",Antonio Vivaldi,Concerto,Orchestral,Orchestra,NaN
10534,"Ellens Gesang I, D.837, op. 52, no. 1, ""Raste,...",Franz Schubert,NaN,Vocal,Voice,NaN
3838,"Fugue in E major, HWV.612",George Frideric Handel,Fugue,Keyboard,Piano,NaN
6966,"Piano Concerto no. 23 in A major, K.488",Wolfgang Amadeus Mozart,Concerto,Orchestral,"Piano, Orchestra",NaN
1898,"Night Mail, film score",Benjamin Britten,NaN,Orchestral,Orchestra,NaN


### Save the Dataset with Feature Engineering

In [16]:
df.to_csv(Path('../data/processed/classical_music_feature_engineered.csv'), index=False)

## Feature Engineering Summary

In this stage, additional features were created from the available metadata to improve the representation of classical musical works for exploratory analysis and the recommendation system.

The following features were extracted and standardized:

* **Form**: Musical forms were extracted from the combination of **Title** and **Subtitle** fields. Different spellings and variations were normalized into consistent categories (e.g., *Sinfonia*, *Sinfonia* → *Symphony*).

* **Extended Genre**: The original genre information was enriched by extracting more specific work types from **Title** and **Subtitle**, including categories such as *Opera*, *Mass*, *Cantata*, *Oratorio*, *Passion*, and *Ballet*. When no additional information was available, the original genre value was retained.

* **Instrumentation / Performing Forces**: Instrument information was extracted from **Title** and **Subtitle**. For works without explicit instrument information, the extended genre and original genre were used to infer likely performing forces.

* **Mode**: Tonality information was analyzed to extract the musical mode (e.g., *Major*, *Minor*) from titles and subtitles. Since mode provides more meaningful information about the overall character of a piece, it was separated from the specific key.

* **Tempo Marking**: Common classical tempo indications (e.g., *Allegro*, *Adagio*, *Andante*) were extracted from titles and subtitles and standardized.

All extracted features were normalized to reduce variations in spelling and terminology. Missing values were kept as `NaN` to distinguish unavailable information from actual categories. These engineered features will be used in the next stages for preprocessing and building the content-based recommendation model.
